In [103]:
import pandas as pd
import seaborn as sns
import numpy as np
import os

df = sns.load_dataset("titanic")

Voici la liste des colonnes où il y a des manquants:

In [104]:
colonnes = ["age", "deck", "embark_town", "embarked"]

for col in colonnes:
    nb_manquants = df[col].isna().sum()
    pourcentage = nb_manquants / len(df) * 100

    print(
        f"{col} : {nb_manquants} valeurs manquantes "
        f"({pourcentage:.2f}%)"
    )

age : 177 valeurs manquantes (19.87%)
deck : 688 valeurs manquantes (77.22%)
embark_town : 2 valeurs manquantes (0.22%)
embarked : 2 valeurs manquantes (0.22%)


Pour la variable « age », nous décidons de remplacer les valeurs manquantes par la médiane de l'âge calculée selon la classe et le sexe.

In [105]:
df["age"].unique()
print("Nombre de valeurs manquantes dans la colonne 'age' avant nettoyage :", df["age"].isna().sum())

df["age"] = df["age"].fillna(df.groupby(["class", "sex"])["age"].transform("median"))

df["age"].unique()
print("Nombre de valeurs manquantes dans la colonne 'age' après nettoyage :", df["age"].isna().sum())

Nombre de valeurs manquantes dans la colonne 'age' avant nettoyage : 177
Nombre de valeurs manquantes dans la colonne 'age' après nettoyage : 0


En analysant la colonne "deck", on constate qu'elle contient près de 80 % de valeurs manquantes. Nous décidons donc de la supprimer et vérifions qu'elle n'est plus présente dans le dataset.

In [106]:
print("Colonnes présentes dans le Dataset :", list(df.columns)
)

df = df.drop(columns=["deck"])

print("Colonnes restantes :", list(df.columns))

Colonnes présentes dans le Dataset : ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']
Colonnes restantes : ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'embark_town', 'alive', 'alone']


Nous savons qu'il y a les colonnes "Embarked" et "Embark_town"sont redondantes, analysons si les valeurs manquantes sont les mêmes:

In [107]:
print("Nombre de valeurs manquantes dans les colonnes 'embark_town' et 'embarked' avant nettoyage :")

for col in ["embark_town", "embarked"]:
    print(
        f"{col} : {df[col].isna().sum()} valeurs manquantes "
        f"({df[col].isna().sum() / len(df) * 100:.2f}%)"
    )

print("\nValeurs manquantes dans les colonnes 'embark_town' et 'embarked' :")
print(df.loc[df["embark_town"].isna() | df["embarked"].isna(), ["embarked", "embark_town"]])

df["embark_town"] = df["embark_town"].fillna("Inconnu")
df["embarked"] = df["embarked"].fillna("Inconnu")

print("\nNombre de valeurs manquantes dans la colonne 'embark_town' et 'embarked' après nettoyage et pourcentage :", (df["embark_town"].isna().sum() + df["embarked"].isna().sum()) / len(df) * 100, "%")
print(df.loc[[61, 829], ["embarked", "embark_town"]])


Nombre de valeurs manquantes dans les colonnes 'embark_town' et 'embarked' avant nettoyage :
embark_town : 2 valeurs manquantes (0.22%)
embarked : 2 valeurs manquantes (0.22%)

Valeurs manquantes dans les colonnes 'embark_town' et 'embarked' :
    embarked embark_town
61       NaN         NaN
829      NaN         NaN

Nombre de valeurs manquantes dans la colonne 'embark_town' et 'embarked' après nettoyage et pourcentage : 0.0 %
    embarked embark_town
61   Inconnu     Inconnu
829  Inconnu     Inconnu


Vu qu'il y a seulement 2 valeurs manquantes, nous avons décider de les remplacer par "Inconnu". Examinons maintenant les doublons:

In [108]:
print("Lignes concernées     :", df.duplicated(keep=False).sum())
print("Doublons en trop      :", df.duplicated().sum())

print("Groupes de doublons   :", df[df.duplicated(keep=False)].groupby(list(df.columns), dropna=False).ngroups)
print("\n")

doublons = df[df.duplicated(keep=False)].sort_values(by=list(df.columns))
for _, bloc in doublons.groupby(list(df.columns), dropna=False):
    display(bloc)
    print("\n")

Lignes concernées     : 175
Doublons en trop      : 118
Groupes de doublons   : 57




,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
263,0,1,male,40.0,0,0,0.0,S,First,man,True,Southampton,no,True
633,0,1,male,40.0,0,0,0.0,S,First,man,True,Southampton,no,True
815,0,1,male,40.0,0,0,0.0,S,First,man,True,Southampton,no,True


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
30,0,1,male,40.0,0,0,27.7208,C,First,man,True,Cherbourg,no,True
64,0,1,male,40.0,0,0,27.7208,C,First,man,True,Cherbourg,no,True
295,0,1,male,40.0,0,0,27.7208,C,First,man,True,Cherbourg,no,True


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
252,0,1,male,62.0,0,0,26.55,S,First,man,True,Southampton,no,True
555,0,1,male,62.0,0,0,26.55,S,First,man,True,Southampton,no,True


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
144,0,2,male,18.0,0,0,11.5,S,Second,man,True,Southampton,no,True
757,0,2,male,18.0,0,0,11.5,S,Second,man,True,Southampton,no,True


...

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
186,1,3,female,21.5,1,0,15.5,Q,Third,woman,False,Queenstown,yes,False
241,1,3,female,21.5,1,0,15.5,Q,Third,woman,False,Queenstown,yes,False
612,1,3,female,21.5,1,0,15.5,Q,Third,woman,False,Queenstown,yes,False


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
347,1,3,female,21.5,1,0,16.1,S,Third,woman,False,Southampton,yes,False
431,1,3,female,21.5,1,0,16.1,S,Third,woman,False,Southampton,yes,False


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
643,1,3,male,25.0,0,0,56.4958,S,Third,man,True,Southampton,yes,True
692,1,3,male,25.0,0,0,56.4958,S,Third,man,True,Southampton,yes,True


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
65,1,3,male,25.0,1,1,15.2458,C,Third,man,True,Cherbourg,yes,False
709,1,3,male,25.0,1,1,15.2458,C,Third,man,True,Cherbourg,yes,False


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
74,1,3,male,32.0,0,0,56.4958,S,Third,man,True,Southampton,yes,True
838,1,3,male,32.0,0,0,56.4958,S,Third,man,True,Southampton,yes,True


Selon moi il ne faut pas les supprimer car nous n'avons pas l'identité des passagers pour affirmer que ce sont des doublons

In [109]:
df.head()


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,Southampton,no,True


On peut constater que les colonnes: "survived" / "alive" ; "pclass" / "class" et "embarked" / "embark_town" sont redondantes.

Les colonnes: "sex", "who", "adult_male" ainsi que "sibsp", "alone, "parch" se marchent dessus.

Après analyse des colonnes, il serait plutôt judicieux selon moi de garder les colonnes suivantes:
embark_town, survived, pclass, who, age, et fare.

Correction: la colonne "sex" m'est utile pour la suite des exercices, je l'enlève donc descolonnes à supprimer

In [110]:
a_supprimer = ['alive', 'class', 'embarked', 'adult_male', 'alone']
df = df.drop(columns=a_supprimer)
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,who,embark_town
0,0,3,male,22.0,1,0,7.2500,man,Southampton
1,1,1,female,38.0,1,0,71.2833,woman,Cherbourg
2,1,3,female,26.0,0,0,7.9250,woman,Southampton
3,1,1,female,35.0,1,0,53.1000,woman,Southampton
4,0,3,male,35.0,0,0,8.0500,man,Southampton


Analysons les types des colonnes:

In [111]:
df.dtypes

survived         int64
pclass           int64
sex                str
age            float64
sibsp            int64
parch            int64
fare           float64
who                str
embark_town        str
dtype: object

Après analyse, il semble pertinent de convertir les variables "who", "pclass" et "embark_town" en type category, car elles représentent des catégories plutôt que des valeurs numériques continues. Bien que la variable "survived" ne possède que deux modalités (0 et 1) et puisse également être convertie en catégorie, nous choisissons de la conserver en int, ce qui facilitera certaines analyses et calculs par la suite.

In [112]:
types_avant = df.dtypes.copy()

colonnes_categories = ['who', 'sex', 'embark_town', 'pclass']
for col in colonnes_categories:
    df[col] = df[col].astype('category')

types_apres = df.dtypes

print(pd.DataFrame({
    "Avant": types_avant,
    "Après": types_apres
}))

               Avant     Après
survived       int64     int64
pclass         int64  category
sex              str  category
age          float64   float64
sibsp          int64     int64
parch          int64     int64
fare         float64   float64
who              str  category
embark_town      str  category


Reperons les billets hors bornes...

Le prix d'un billet dépend de plusieurs parametres: de l'age, du sexe / profil, de la classe ainsi que du port d'embarquement.

In [113]:
print(df[['age', 'who', 'sex','pclass', 'embark_town', 'fare']].describe(include='all'))

               age  who   sex  pclass  embark_town        fare
count   891.000000  891   891   891.0          891  891.000000
unique         NaN    3     2     3.0            4         NaN
top            NaN  man  male     3.0  Southampton         NaN
freq           NaN  537   577   491.0          644         NaN
mean     29.112424  NaN   NaN     NaN          NaN   32.204208
std      13.304424  NaN   NaN     NaN          NaN   49.693429
min       0.420000  NaN   NaN     NaN          NaN    0.000000
25%      21.500000  NaN   NaN     NaN          NaN    7.910400
50%      26.000000  NaN   NaN     NaN          NaN   14.454200
75%      36.000000  NaN   NaN     NaN          NaN   31.000000
max      80.000000  NaN   NaN     NaN          NaN  512.329200


Analysons la distribution des tarifs au sein de chaque groupe avant de rechercher les valeurs atypiques:

In [114]:
display(
    df.groupby(['pclass', 'embark_town', 'who'])['fare']
      .describe()
      .round(2)
)

count    mean     std    min     25%     50%  \
pclass embark_town who                                                   
1      Cherbourg   man     42.0   93.54  108.76  26.55   30.70   61.68   
                   woman   43.0  115.64   88.57  27.72   69.30   83.16   
       Inconnu     woman    2.0   80.00    0.00  80.00   80.00   80.00   
       Queenstown  man      1.0   90.00     NaN  90.00   90.00   90.00   
                   woman    1.0   90.00     NaN  90.00   90.00   90.00   
       Southampton child    6.0  139.38   43.67  81.86  120.00  135.78   
                   man     76.0   50.39   49.18   0.00   26.55   33.76   
                   woman   45.0   94.90   60.09  25.93   53.10   78.85   
2      Cherbourg   child    3.0   36.22    5.79  30.07   33.54   37.00   
                   man      9.0   24.13   10.70  12.88   15.05   24.00   
                   woman    5.0   21.05   12.41  12.00   13.79   13.86   
       Queenstown  man      1.0   12.35     NaN  12.35   12.35   12.35   
                   woman    2.0   12.35    0.00  12.35   12.35   12.35   
       Southampton child   16.0   26.84    7.27  14.50   22.12   26.12   
                   man     89.0   18.62   15.60   0.00   11.50   13.00   
                   woman   59.0   21.14   11.06  10.50   13.00   21.00   
3      Cherbourg   child   15.0   13.50    4.55   7.22    9.88   14.45   
                   man     39.0    9.14    3.89   4.01    7.22    7.23   
                   woman   12.0   15.10    4.76   7.22   14.45   14.46   
       Queenstown  child    5.0   24.91    9.43   8.03   29.12   29.12   
                   man     35.0    9.96    5.00   6.75    7.75    7.75   
                   woman   32.0   10.38    5.64   6.75    7.75    7.75   
       Southampton child   38.0   26.83   11.38   7.85   17.66   27.90   
                   man    245.0   11.89   11.87   0.00    7.80    7.92   
                   woman   70.0   17.67   14.60   7.25    7.92   10.82   

                             75%     max  
pclass embark_town who                    
1      Cherbourg   man    102.59  512.33  
                   woman  134.50  512.33  
       Inconnu     woman   80.00   80.00  
       Queenstown  man     90.00   90.00  
                   woman   90.00   90.00  
       Southampton child  151.55  211.34  
                   man     53.10  263.00  
                   woman  133.65  263.00  
2      Cherbourg   child   39.29   41.58  
                   man     30.07   41.58  
                   woman   24.00   41.58  
       Queenstown  man     12.35   12.35  
                   woman   12.35   12.35  
       Southampton child   30.00   39.00  
                   man     26.00   73.50  
                   woman   26.00   65.00  
3      Cherbourg   child   17.26   19.26  
                   man      8.28   21.68  
                   woman   16.25   22.36  
       Queenstown  child   29.12   29.12  
                   man      7.79   24.15  
                   woman    7.94   29.12  
       Southampton child   31.39   46.90  
                   man      9.00   69.55  
                   woman   20.46   69.55

Une borne haute est calculée pour chaque groupe. Si le prix la dépasse cette borne, c'est considéré comme "hors bornes" avec un ratio prix / borne haute.

In [115]:
def borne_haute(x):
    q1 = x.quantile(0.25)
    q3 = x.quantile(0.75)
    return q3 + 1.5 * (q3 - q1)

df['borne_haute'] = (
    df.groupby(['pclass', 'embark_town', 'who'])['fare']
      .transform(borne_haute)
)

hors_bornes = df[df['fare'] > df['borne_haute']].copy()
hors_bornes['ratio'] = hors_bornes['fare'] / hors_bornes['borne_haute']

print(f"Nombre de billets hors bornes : {len(hors_bornes)}")

pd.set_option('display.max_rows', None)
for groupe, data in hors_bornes.groupby(['pclass', 'embark_town', 'who']):
    print(f"\n\n {groupe} \n")
    display(
        data[['age', 'fare', 'borne_haute', 'ratio']]
        .sort_values('fare', ascending=False)
        .round(2)
    )

Nombre de billets hors bornes : 103


 (1, 'Cherbourg', 'man') 



,age,fare,borne_haute,ratio
737,35.0,512.33,210.43,2.43
679,36.0,512.33,210.43,2.43
118,24.0,247.52,210.43,1.18
557,40.0,227.52,210.43,1.08
377,27.0,211.50,210.43,1.01




 (1, 'Cherbourg', 'woman') 



,age,fare,borne_haute,ratio
258,35.0,512.33,232.3,2.21
311,18.0,262.38,232.3,1.13
742,21.0,262.38,232.3,1.13
299,50.0,247.52,232.3,1.07




 (1, 'Southampton', 'child') 



...

,age,fare,borne_haute,ratio
885,39.0,29.12,8.23,3.54
109,21.5,24.15,8.23,2.93
330,21.5,23.25,8.23,2.82
241,21.5,15.50,8.23,1.88
186,21.5,15.50,8.23,1.88
612,21.5,15.50,8.23,1.88
657,32.0,15.50,8.23,1.88




 (3, 'Southampton', 'man') 



,age,fare,borne_haute,ratio
159,25.0,69.55,10.81,6.44
201,25.0,69.55,10.81,6.44
324,25.0,69.55,10.81,6.44
846,25.0,69.55,10.81,6.44
169,28.0,56.50,10.81,5.23
509,26.0,56.50,10.81,5.23
838,32.0,56.50,10.81,5.23
826,25.0,56.50,10.81,5.23
643,25.0,56.50,10.81,5.23
74,32.0,56.50,10.81,5.23




 (3, 'Southampton', 'woman') 



,age,fare,borne_haute,ratio
180,21.5,69.55,39.25,1.77
863,21.5,69.55,39.25,1.77
792,21.5,69.55,39.25,1.77
71,16.0,46.90,39.25,1.19
678,43.0,46.90,39.25,1.19
638,41.0,39.69,39.25,1.01


Nous allons maintenant répartir ces différents prix hors_bornes en foction de leur ratio. Nous savons également que plusieurs voyageurs peuvent partager le même billet de train, ce qui pourrait expliquer un prix élever 

In [116]:
hors_bornes = hors_bornes.copy()

hors_bornes['personnes_billet'] = hors_bornes['sibsp'] + hors_bornes['parch'] + 1
hors_bornes['prix_moyen'] = hors_bornes['fare'] / hors_bornes['personnes_billet']

hors_bornes['explication'] = np.where(
    hors_bornes['personnes_billet'] > 1,
    'Billet possiblement partagé',
    'Voyageur seul'
)

cols = ['pclass', 'embark_town', 'who', 'age', 'personnes_billet','explication', 'ratio', 'prix_moyen', 'fare', 'borne_haute', ]

print("Répartition des billets hors bornes selon le ratio\n")

print(f"\n\nDépassement léger (1 <= ratio < 1.5) : {((hors_bornes['ratio'] >= 1) & (hors_bornes['ratio'] < 1.5)).sum()}\n")
display(
    hors_bornes[(hors_bornes['ratio'] >= 1) & (hors_bornes['ratio'] < 1.5)][cols]
    .sort_values('ratio', ascending=False)
    .round(2)
)

print(f"\n\nDépassement modéré (1.5 <= ratio < 2) : {((hors_bornes['ratio'] >= 1.5) & (hors_bornes['ratio'] < 2)).sum()}\n")
display(
    hors_bornes[(hors_bornes['ratio'] >= 1.5) & (hors_bornes['ratio'] < 2)][cols]
    .sort_values('ratio', ascending=False)
    .round(2)
)

print(f"\n\nDépassement important (2 <= ratio < 2.5) : {((hors_bornes['ratio'] >= 2) & (hors_bornes['ratio'] < 2.5)).sum()}\n")
display(
    hors_bornes[(hors_bornes['ratio'] >= 2) & (hors_bornes['ratio'] < 2.5)][cols]
    .sort_values('ratio', ascending=False)
    .round(2)
)

print(f"\n\nDépassement très important (2.5 <= ratio < 3) : {((hors_bornes['ratio'] >= 2.5) & (hors_bornes['ratio'] < 3)).sum()}\n")
display(
    hors_bornes[(hors_bornes['ratio'] >= 2.5) & (hors_bornes['ratio'] < 3)][cols]
    .sort_values('ratio', ascending=False)
    .round(2)
)

print(f"\n\nDépassement extrême (ratio >= 3) : {(hors_bornes['ratio'] >= 3).sum()}\n")
display(
    hors_bornes[hors_bornes['ratio'] >= 3][cols]
    .sort_values('ratio', ascending=False)
    .round(2)
)

Répartition des billets hors bornes selon le ratio



Dépassement léger (1 <= ratio < 1.5) : 40



,pclass,embark_town,who,age,personnes_billet,explication,ratio,prix_moyen,fare,borne_haute
160,3,Southampton,man,44.0,2,Billet possiblement partagé,1.49,8.05,16.10,10.81
639,3,Southampton,man,25.0,2,Billet possiblement partagé,1.49,8.05,16.10,10.81
743,3,Southampton,man,24.0,2,Billet possiblement partagé,1.49,8.05,16.10,10.81
735,3,Southampton,man,28.5,1,Voyageur seul,1.49,16.10,16.10,10.81
624,3,Southampton,man,21.0,1,Voyageur seul,1.49,16.10,16.10,10.81
253,3,Southampton,man,30.0,2,Billet possiblement partagé,1.49,8.05,16.10,10.81
206,3,Southampton,man,32.0,2,Billet possiblement partagé,1.47,7.92,15.85,10.81
403,3,Southampton,man,28.0,2,Billet possiblement partagé,1.47,7.92,15.85,10.81
495,3,Cherbourg,man,25.0,1,Voyageur seul,1.47,14.46,14.46,9.86
620,3,Cherbourg,man,27.0,2,Billet possiblement partagé,1.47,7.23,14.45,9.86




Dépassement modéré (1.5 <= ratio < 2) : 29



,pclass,embark_town,who,age,personnes_billet,explication,ratio,prix_moyen,fare,borne_haute
46,3,Queenstown,man,25.0,2,Billet possiblement partagé,1.97,7.75,15.50,7.86
188,3,Queenstown,man,40.0,3,Billet possiblement partagé,1.97,5.17,15.50,7.86
364,3,Queenstown,man,25.0,2,Billet possiblement partagé,1.97,7.75,15.50,7.86
718,3,Queenstown,man,25.0,1,Voyageur seul,1.97,15.50,15.50,7.86
207,3,Cherbourg,man,26.0,1,Voyageur seul,1.91,18.79,18.79,9.86
93,3,Southampton,man,26.0,4,Billet possiblement partagé,1.90,5.14,20.58,10.81
548,3,Southampton,man,33.0,3,Billet possiblement partagé,1.90,6.84,20.52,10.81
241,3,Queenstown,woman,21.5,2,Billet possiblement partagé,1.88,7.75,15.50,8.23
657,3,Queenstown,woman,32.0,3,Billet possiblement partagé,1.88,5.17,15.50,8.23
612,3,Queenstown,woman,21.5,2,Billet possiblement partagé,1.88,7.75,15.50,8.23




Dépassement important (2 <= ratio < 2.5) : 11



,pclass,embark_town,who,age,personnes_billet,explication,ratio,prix_moyen,fare,borne_haute
737,1,Cherbourg,man,35.0,1,Voyageur seul,2.43,512.33,512.33,210.43
679,1,Cherbourg,man,36.0,2,Billet possiblement partagé,2.43,256.16,512.33,210.43
527,1,Southampton,man,40.0,1,Voyageur seul,2.39,221.78,221.78,92.93
176,3,Southampton,man,25.0,5,Billet possiblement partagé,2.36,5.09,25.47,10.81
595,3,Southampton,man,36.0,3,Billet possiblement partagé,2.23,8.05,24.15,10.81
811,3,Southampton,man,39.0,1,Voyageur seul,2.23,24.15,24.15,10.81
565,3,Southampton,man,24.0,3,Billet possiblement partagé,2.23,8.05,24.15,10.81
258,1,Cherbourg,woman,35.0,1,Voyageur seul,2.21,512.33,512.33,232.30
48,3,Cherbourg,man,25.0,3,Billet possiblement partagé,2.20,7.23,21.68,9.86
783,3,Southampton,man,25.0,4,Billet possiblement partagé,2.17,5.86,23.45,10.81




Dépassement très important (2.5 <= ratio < 3) : 7



,pclass,embark_town,who,age,personnes_billet,explication,ratio,prix_moyen,fare,borne_haute
301,3,Queenstown,man,25.0,3,Billet possiblement partagé,2.96,7.75,23.25,7.86
109,3,Queenstown,woman,21.5,2,Billet possiblement partagé,2.93,12.08,24.15,8.23
13,3,Southampton,man,39.0,7,Billet possiblement partagé,2.89,4.47,31.28,10.81
27,1,Southampton,man,19.0,6,Billet possiblement partagé,2.83,43.83,263.00,92.93
438,1,Southampton,man,64.0,6,Billet possiblement partagé,2.83,43.83,263.00,92.93
330,3,Queenstown,woman,21.5,3,Billet possiblement partagé,2.82,7.75,23.25,8.23
360,3,Southampton,man,40.0,6,Billet possiblement partagé,2.58,4.65,27.90,10.81




Dépassement extrême (ratio >= 3) : 16



,pclass,embark_town,who,age,personnes_billet,explication,ratio,prix_moyen,fare,borne_haute
159,3,Southampton,man,25.0,11,Billet possiblement partagé,6.44,6.32,69.55,10.81
201,3,Southampton,man,25.0,11,Billet possiblement partagé,6.44,6.32,69.55,10.81
324,3,Southampton,man,25.0,11,Billet possiblement partagé,6.44,6.32,69.55,10.81
846,3,Southampton,man,25.0,11,Billet possiblement partagé,6.44,6.32,69.55,10.81
509,3,Southampton,man,26.0,1,Voyageur seul,5.23,56.50,56.50,10.81
169,3,Southampton,man,28.0,1,Voyageur seul,5.23,56.50,56.50,10.81
643,3,Southampton,man,25.0,1,Voyageur seul,5.23,56.50,56.50,10.81
74,3,Southampton,man,32.0,1,Voyageur seul,5.23,56.50,56.50,10.81
838,3,Southampton,man,32.0,1,Voyageur seul,5.23,56.50,56.50,10.81
692,3,Southampton,man,25.0,1,Voyageur seul,5.23,56.50,56.50,10.81


Analysons maintenant le nombre d'anomalies en fonction de prix moyen par rapport aux nombre de passagers comptabilisés sur le billet pour ceux voyageant à plusieurs:

In [117]:

df['personnes_billet'] = df['sibsp'] + df['parch'] + 1
df['prix_moyen'] = df['fare'] / df['personnes_billet']

df['borne_haute_moyen'] = (
    df.groupby(['pclass', 'embark_town', 'who'])['prix_moyen']
      .transform(borne_haute)
)

anomalies_corrigees = df[
    (df['fare'] > df['borne_haute'])
    & (df['prix_moyen'] > df['borne_haute_moyen'])
].copy()


anomalies_corrigees['ratio_moyen'] = (
    anomalies_corrigees['prix_moyen']
    / anomalies_corrigees['borne_haute_moyen']
)

anomalies_corrigees['explication'] = np.where(
    anomalies_corrigees['personnes_billet'] > 1,
    'Billet possiblement partagé',
    'Voyageur seul'
)

cols = [
    'pclass',
    'embark_town',
    'who',
    'age',
    'personnes_billet',
    'explication',
    'ratio_moyen',
    'prix_moyen',
    'borne_haute_moyen',
    'fare'
]

print(
    f"Nombre d'anomalies confirmées (hors borne sur le prix du billet "
    f"et sur le prix par voyageur) : {len(anomalies_corrigees)}\n"
)

print(f"\n\nDépassement léger (1 <= ratio < 1.5) : {((anomalies_corrigees['ratio_moyen'] >= 1) & (anomalies_corrigees['ratio_moyen'] < 1.5)).sum()}\n")
display(
    anomalies_corrigees[
        (anomalies_corrigees['ratio_moyen'] >= 1)
        & (anomalies_corrigees['ratio_moyen'] < 1.5)
    ][cols]
    .sort_values('ratio_moyen', ascending=False)
    .round(2)
)

print(f"\n\nDépassement modéré (1.5 <= ratio < 2) : {((anomalies_corrigees['ratio_moyen'] >= 1.5) & (anomalies_corrigees['ratio_moyen'] < 2)).sum()}\n")
display(
    anomalies_corrigees[
        (anomalies_corrigees['ratio_moyen'] >= 1.5)
        & (anomalies_corrigees['ratio_moyen'] < 2)
    ][cols]
    .sort_values('ratio_moyen', ascending=False)
    .round(2)
)

print(f"\n\nDépassement important (2 <= ratio < 2.5) : {((anomalies_corrigees['ratio_moyen'] >= 2) & (anomalies_corrigees['ratio_moyen'] < 2.5)).sum()}\n")
display(
    anomalies_corrigees[
        (anomalies_corrigees['ratio_moyen'] >= 2)
        & (anomalies_corrigees['ratio_moyen'] < 2.5)
    ][cols]
    .sort_values('ratio_moyen', ascending=False)
    .round(2)
)

print(f"\n\nDépassement très important (2.5 <= ratio < 3) : {((anomalies_corrigees['ratio_moyen'] >= 2.5) & (anomalies_corrigees['ratio_moyen'] < 3)).sum()}\n")
display(
    anomalies_corrigees[
        (anomalies_corrigees['ratio_moyen'] >= 2.5)
        & (anomalies_corrigees['ratio_moyen'] < 3)
    ][cols]
    .sort_values('ratio_moyen', ascending=False)
    .round(2)
)

print(f"\n\nDépassement extrême (ratio >= 3) : {(anomalies_corrigees['ratio_moyen'] >= 3).sum()}\n")
display(
    anomalies_corrigees[
        anomalies_corrigees['ratio_moyen'] >= 3
    ][cols]
    .sort_values('ratio_moyen', ascending=False)
    .round(2)
)

Nombre d'anomalies confirmées (hors borne sur le prix du billet et sur le prix par voyageur) : 37



Dépassement léger (1 <= ratio < 1.5) : 7



,pclass,embark_town,who,age,personnes_billet,explication,ratio_moyen,prix_moyen,borne_haute_moyen,fare
120,2,Southampton,man,21.0,3,Billet possiblement partagé,1.46,24.50,16.75,73.50
655,2,Southampton,man,24.0,3,Billet possiblement partagé,1.46,24.50,16.75,73.50
665,2,Southampton,man,32.0,3,Billet possiblement partagé,1.46,24.50,16.75,73.50
118,1,Cherbourg,man,24.0,2,Billet possiblement partagé,1.44,123.76,86.05,247.52
5,3,Queenstown,man,25.0,1,Voyageur seul,1.09,8.46,7.77,8.46
490,3,Southampton,man,25.0,2,Billet possiblement partagé,1.08,9.98,9.25,19.97
451,3,Southampton,man,25.0,2,Billet possiblement partagé,1.08,9.98,9.25,19.97




Dépassement modéré (1.5 <= ratio < 2) : 11



,pclass,embark_town,who,age,personnes_billet,explication,ratio_moyen,prix_moyen,borne_haute_moyen,fare
495,3,Cherbourg,man,25.0,1,Voyageur seul,2.00,14.46,7.24,14.46
718,3,Queenstown,man,25.0,1,Voyageur seul,1.99,15.50,7.77,15.50
624,3,Southampton,man,21.0,1,Voyageur seul,1.74,16.10,9.25,16.10
735,3,Southampton,man,28.5,1,Voyageur seul,1.74,16.10,9.25,16.10
497,3,Southampton,man,25.0,1,Voyageur seul,1.63,15.10,9.25,15.10
575,3,Southampton,man,19.0,1,Voyageur seul,1.57,14.50,9.25,14.50
538,3,Southampton,man,25.0,1,Voyageur seul,1.57,14.50,9.25,14.50
760,3,Southampton,man,25.0,1,Voyageur seul,1.57,14.50,9.25,14.50
332,1,Southampton,man,38.0,2,Billet possiblement partagé,1.56,76.73,49.21,153.46
768,3,Queenstown,man,25.0,2,Billet possiblement partagé,1.55,12.08,7.77,24.15




Dépassement important (2 <= ratio < 2.5) : 2



,pclass,embark_town,who,age,personnes_billet,explication,ratio_moyen,prix_moyen,borne_haute_moyen,fare
508,3,Southampton,man,28.0,1,Voyageur seul,2.44,22.52,9.25,22.52
689,1,Southampton,child,15.0,2,Billet possiblement partagé,2.13,105.67,49.72,211.34




Dépassement très important (2.5 <= ratio < 3) : 4



,pclass,embark_town,who,age,personnes_billet,explication,ratio_moyen,prix_moyen,borne_haute_moyen,fare
679,1,Cherbourg,man,36.0,2,Billet possiblement partagé,2.98,256.16,86.05,512.33
557,1,Cherbourg,man,40.0,1,Voyageur seul,2.64,227.52,86.05,227.52
811,3,Southampton,man,39.0,1,Voyageur seul,2.61,24.15,9.25,24.15
207,3,Cherbourg,man,26.0,1,Voyageur seul,2.60,18.79,7.24,18.79




Dépassement extrême (ratio >= 3) : 13



,pclass,embark_town,who,age,personnes_billet,explication,ratio_moyen,prix_moyen,borne_haute_moyen,fare
74,3,Southampton,man,32.0,1,Voyageur seul,6.11,56.50,9.25,56.50
169,3,Southampton,man,28.0,1,Voyageur seul,6.11,56.50,9.25,56.50
826,3,Southampton,man,25.0,1,Voyageur seul,6.11,56.50,9.25,56.50
643,3,Southampton,man,25.0,1,Voyageur seul,6.11,56.50,9.25,56.50
509,3,Southampton,man,26.0,1,Voyageur seul,6.11,56.50,9.25,56.50
838,3,Southampton,man,32.0,1,Voyageur seul,6.11,56.50,9.25,56.50
692,3,Southampton,man,25.0,1,Voyageur seul,6.11,56.50,9.25,56.50
737,1,Cherbourg,man,35.0,1,Voyageur seul,5.95,512.33,86.05,512.33
527,1,Southampton,man,40.0,1,Voyageur seul,4.51,221.78,49.21,221.78
72,2,Southampton,man,21.0,1,Voyageur seul,4.39,73.50,16.75,73.50


In [118]:
df['ratio_moyen'] = df['prix_moyen'] / df['borne_haute_moyen']


df['hors_borne_fare'] = df['fare'] > df['borne_haute']


df['hors_borne_ajuste'] = df['prix_moyen'] > df['borne_haute_moyen']


df['hors_borne'] = df['hors_borne_fare'] & df['hors_borne_ajuste']
df['explique_partage'] = df['hors_borne_fare'] & ~df['hors_borne_ajuste']

df['niveau_anomalie'] = np.select(
    [
        df['hors_borne'] & (df['ratio_moyen'] < 1.5),
        df['hors_borne'] & (df['ratio_moyen'] >= 1.5) & (df['ratio_moyen'] < 2),
        df['hors_borne'] & (df['ratio_moyen'] >= 2) & (df['ratio_moyen'] < 2.5),
        df['hors_borne'] & (df['ratio_moyen'] >= 2.5) & (df['ratio_moyen'] < 3),
        df['hors_borne'] & (df['ratio_moyen'] >= 3),
    ],
    ['Léger', 'Modéré', 'Important', 'Très important', 'Extrême'],
    default='Aucune'
)

assert df['hors_borne_fare'].sum() == len(hors_bornes)

anomalie = ['Aucune', 'Léger', 'Modéré', 'Important', 'Très important', 'Extrême']
print("Repérés sur le prix du billet     :", df['hors_borne_fare'].sum())
print("Expliqués par un billet partagé   :", df['explique_partage'].sum())
print("Hors bornes confirmés             :", df['hors_borne'].sum())
print()
display(df['niveau_anomalie'].value_counts().reindex(anomalie).to_frame('Hors bornes'))
print("\nLignes supprimées :", 0)        

Repérés sur le prix du billet     : 103
Expliqués par un billet partagé   : 66
Hors bornes confirmés             : 37



,Hors bornes
niveau_anomalie,
Aucune,854
Léger,7
Modéré,11
Important,2
Très important,4
Extrême,13



Lignes supprimées : 0


**Decision 1 : deck supprimée**
- **Quoi** : suppression de la colonne `deck`
- **Pourquoi** : 77 % de valeurs manquantes, pas fiable
- **Lignes** : 891

**Décision 2 : age modifié**
- **Quoi** : remplacement des manquants par la médiane de l'âge selon la classe et le sexe
- **Pourquoi** : l'âge varie fortement selon ces deux variables, la médiane nous permet d'ajuster
- **Lignes** : 177

**Décision 3 : embark_town et embarked complétés**
- **Quoi** : remplacement des manquants par "Inconnu"
- **Pourquoi** : seulement 2 manquants
- **Lignes** : 2

**Décision 4 : doublons conservés**
- **Quoi** : aucune suppression
- **Pourquoi** : pas d'identifiant passager, deux voyageurs différents peuvent avoir des caractéristiques identiques
- **Lignes** : 175 lignes concernées, 0 supprimées

**Décision 5 : colonnes redondantes supprimées**
- **Quoi** : suppression de `alive`, `class`, `embarked`, `sex`, `adult_male`, `alone`
- **Pourquoi** : elles dupliquent `survived`, `pclass`, `embark_town`, `who`, `sibsp`/`parch`
- **Lignes** : 891
(Correction: Rajout de la colonne "sex", utile pour la suite du projet)

**Décision 6 : types corrigés**
- **Quoi** : `who`, `embark_town`, `pclass` convertis en `category`. `survived` conservé en `int`
- **Pourquoi** : ce sont des modalités et non des mesures continues, `survived` reste numérique pour faciliter les calculs
- **Lignes** : 891

**Décision 7 : billets hors bornes conservés et signalés**
- **Quoi** : colonne `fare`, aucune suppression
- **Pourquoi** : 103 billets dépassent la borne haute. Pour 66 ça redevient normal car "billet partagé". 37 hors bornes mais tarifs explicables: un voyageur classé « seul » peut partager un billet avec des personnes sans lien familial, peut être frais supplémentaires...
- **Lignes** : 37 signalées, 66 expliquées par un billet partagé, 0 supprimée


In [119]:
print(list(df.columns))

colonnes_aux = [
    'borne_haute', 'personnes_billet', 'prix_moyen', 'borne_haute_moyen',
    'ratio_moyen', 'hors_borne_fare', 'hors_borne_ajuste',
    'hors_borne', 'explique_partage'
]

os.makedirs('../data', exist_ok=True)
df.drop(columns=colonnes_aux).to_csv('../data/titanic_propre.csv', index=False)
print("CSV enregistré dans :", os.path.abspath('../data/titanic_propre.csv'))

['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'who', 'embark_town', 'borne_haute', 'personnes_billet', 'prix_moyen', 'borne_haute_moyen', 'ratio_moyen', 'hors_borne_fare', 'hors_borne_ajuste', 'hors_borne', 'explique_partage', 'niveau_anomalie']
CSV enregistré dans : c:\Users\D\Documents\Cours\HETIC_2eme_annee\Python - Analyse de données\pyhton-exercices\data\titanic_propre.csv
